# 04 · Pipeline Test: MobileNetV3-Large on One Fold

**Goal:** check the whole training pipeline end to end **before** running the full benchmark (6 models x 5 folds):
load data from `splits.csv` → augment → train → evaluate → measure size and speed.

**Setup for this test**
- Model: **MobileNetV3-Large**, ImageNet-pretrained, with a new binary head.
- Data: only `keep == True` rows (one file per exact copy). **Test = fold 0**, **validation = fold 1**, **training = folds 2-4**.
- Training in two phases: (1) freeze the backbone and train only the head; (2) unfreeze everything and fine-tune with a small learning rate.
- Early stopping watches **validation loss**, because validation AUC saturates at 1.0 and stops being informative. If fine-tuning never beats the head-only model, we keep the head-only weights.
- The **decision threshold** is chosen on the validation fold and then applied unchanged to the test fold.

**Accelerator:** GPU (T4 or P100).

In [ ]:
import json, os, random, time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import keras
from keras import layers
from PIL import Image
from sklearn.metrics import (accuracy_score, confusion_matrix, f1_score, precision_score,
                             recall_score, roc_auc_score, roc_curve)

# ---------------- settings ----------------
MODEL_NAME  = "MobileNetV3Large"
TEST_FOLD   = 0
VAL_FOLD    = 1
IMG_SIZE    = 224
BATCH       = 32
EPOCHS_HEAD = 5       # phase 1: backbone frozen
EPOCHS_FT   = 30      # phase 2: fine-tune all layers (early stopping usually ends sooner)
LR_HEAD     = 1e-3
LR_FT       = 1e-5     # small: with only ~430 training images, larger rates overfit (seen in v1)
PATIENCE    = 7
SEED        = 42
# ------------------------------------------

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); tf.random.set_seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
set_seed(SEED)

PATCHES = next(Path("/kaggle/input").rglob("Patches"))
SPLITS = next(Path("/kaggle/input").rglob("splits.csv"))
print("TensorFlow", tf.__version__, "| Keras", keras.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))
print("Patches:", PATCHES, "\nSplits:", SPLITS)

## 1. Load the split and the images

In [ ]:
splits = pd.read_csv(SPLITS)
data = splits[splits.keep].reset_index(drop=True)

def role(f):
    return "test" if f == TEST_FOLD else "val" if f == VAL_FOLD else "train"
data["role"] = data.fold.map(role)

def load(row):
    im = Image.open(PATCHES / row["class"] / row.file).convert("RGB").resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)
    return np.asarray(im, dtype=np.uint8)

X = np.stack([load(r) for _, r in data.iterrows()])
y = data.label.values.astype(np.float32)          # 1 = ulcer, 0 = normal
idx = {r: np.where(data.role == r)[0] for r in ["train", "val", "test"]}

summary = data.groupby("role").label.agg(images="size", ulcer="sum")
summary["normal"] = summary.images - summary.ulcer
print("Images in memory:", X.shape)
summary

## 2. Input pipeline and augmentation
Augmentation is applied to the **training set only**. MobileNetV3 in Keras includes its own preprocessing, so images go in as 0-255 pixels.

In [ ]:
augment = keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.15, fill_mode="reflect"),
    layers.RandomZoom(0.1, fill_mode="reflect"),
    layers.RandomContrast(0.15),
    layers.RandomBrightness(0.1, value_range=(0, 255)),
], name="augment")

def make_ds(ids, training):
    ds = tf.data.Dataset.from_tensor_slices((X[ids].astype(np.float32), y[ids]))
    if training:
        ds = ds.shuffle(len(ids), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.batch(BATCH)
    if training:
        ds = ds.map(lambda a, b: (augment(a, training=True), b), num_parallel_calls=tf.data.AUTOTUNE)
    return ds.prefetch(tf.data.AUTOTUNE)

train_ds, val_ds, test_ds = make_ds(idx["train"], True), make_ds(idx["val"], False), make_ds(idx["test"], False)

# Show a few augmented training images
batch = next(iter(train_ds))
plt.figure(figsize=(12, 3.2))
for i in range(8):
    plt.subplot(1, 8, i + 1); plt.imshow(batch[0][i].numpy().clip(0, 255).astype("uint8"))
    plt.title("ulcer" if batch[1][i] == 1 else "normal", fontsize=9); plt.axis("off")
plt.suptitle("Augmented training samples"); plt.tight_layout(); plt.show()

## 3. Model
Pretrained backbone → global average pooling → dropout → 1 sigmoid output. The backbone's BatchNorm layers are kept frozen during fine-tuning, so their statistics stay fixed, which is standard for small datasets.

In [ ]:
BACKBONES = {
    "MobileNetV3Large": lambda: keras.applications.MobileNetV3Large(
        include_top=False, weights="imagenet", input_shape=(IMG_SIZE, IMG_SIZE, 3), include_preprocessing=True),
}

def build_model(name):
    base = BACKBONES[name]()
    base.trainable = False
    inp = keras.Input((IMG_SIZE, IMG_SIZE, 3))
    x = base(inp, training=False)
    x = layers.GlobalAveragePooling2D(name="gap")(x)
    x = layers.Dropout(0.3, name="dropout")(x)
    out = layers.Dense(1, activation="sigmoid", name="classifier")(x)
    return keras.Model(inp, out, name=name), base

model, base = build_model(MODEL_NAME)
print(f"Total parameters: {model.count_params():,}")

## 4. Training
Class weights balance the ulcer/normal ratio in the training folds.

In [ ]:
ytr = y[idx["train"]]
class_weight = {0: len(ytr) / (2 * (ytr == 0).sum()), 1: len(ytr) / (2 * (ytr == 1).sum())}
print("Class weights:", {k: round(v, 3) for k, v in class_weight.items()})

def compile_model(lr):
    model.compile(optimizer=keras.optimizers.Adam(lr), loss="binary_crossentropy",
                  metrics=[keras.metrics.AUC(name="auc"), keras.metrics.BinaryAccuracy(name="acc")])

t0 = time.time()
# Phase 1: train the head only
compile_model(LR_HEAD)
h1 = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_HEAD, class_weight=class_weight, verbose=2)
head_weights = model.get_weights()
head_val_loss = model.evaluate(val_ds, verbose=0)[0]

# Phase 2: fine-tune the whole network, but keep BatchNorm layers frozen.
# (Tested: in Keras 3, unfrozen BatchNorm layers switch to training mode even when called with
#  training=False, which made the training loss jump at the start of fine-tuning in v2.)
base.trainable = True
for layer in base.layers:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False
steps = EPOCHS_FT * len(train_ds)
compile_model(keras.optimizers.schedules.CosineDecay(LR_FT, decay_steps=steps))
stop = keras.callbacks.EarlyStopping(monitor="val_loss", mode="min", patience=PATIENCE, restore_best_weights=True)
h2 = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_FT, class_weight=class_weight,
               callbacks=[stop], verbose=2)
ft_val_loss = min(h2.history["val_loss"])

# Keep whichever phase generalised better on the validation fold
if ft_val_loss < head_val_loss:
    chosen = f"fine-tuned (best epoch {int(np.argmin(h2.history['val_loss'])) + 1})"
else:
    model.set_weights(head_weights)
    chosen = "head-only (fine-tuning did not lower validation loss)"
train_time = time.time() - t0
print(f"Validation loss: head-only {head_val_loss:.4f} | fine-tuned best {ft_val_loss:.4f} -> using {chosen}")
print(f"Training time: {train_time / 60:.1f} min")

In [ ]:
hist = {k: h1.history[k] + h2.history[k] for k in h1.history}
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
for k in ["loss", "val_loss"]: ax[0].plot(hist[k], label=k)
for k in ["auc", "val_auc"]: ax[1].plot(hist[k], label=k)
for a, t in zip(ax, ["Loss", "AUC"]):
    a.axvline(EPOCHS_HEAD - 0.5, color="grey", ls=":", label="start fine-tuning"); a.set_title(t); a.set_xlabel("epoch"); a.legend()
plt.tight_layout(); plt.show()

## 5. Evaluation on the held-out test fold

In [ ]:
# Choose the decision threshold on the VALIDATION fold (never on test):
# maximise balanced accuracy; if several thresholds tie, take the middle of that range.
p_val, y_val = model.predict(val_ds, verbose=0).ravel(), y[idx["val"]]
grid = np.linspace(0.01, 0.99, 99)
bal = np.array([((p_val >= t)[y_val == 1].mean() + (p_val < t)[y_val == 0].mean()) / 2 for t in grid])
THRESHOLD = float(np.median(grid[bal == bal.max()]))
print(f"Threshold chosen on validation: {THRESHOLD:.2f} (validation balanced accuracy {bal.max():.3f})")

p_test = model.predict(test_ds, verbose=0).ravel()
y_test = y[idx["test"]]
pred = (p_test >= THRESHOLD).astype(int)
acc_at_05 = accuracy_score(y_test, (p_test >= 0.5).astype(int))
tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()
metrics = {
    "accuracy": accuracy_score(y_test, pred),
    "precision": precision_score(y_test, pred),
    "sensitivity": recall_score(y_test, pred),
    "specificity": tn / (tn + fp),
    "f1": f1_score(y_test, pred),
    "auc": roc_auc_score(y_test, p_test),
}
print(pd.Series(metrics).round(4).to_string())
print(f"(for reference, accuracy with a fixed 0.5 threshold: {acc_at_05:.4f})")

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
cm = np.array([[tn, fp], [fn, tp]])
ax[0].imshow(cm, cmap="Blues")
for (i, j), v in np.ndenumerate(cm): ax[0].text(j, i, v, ha="center", va="center", fontsize=14)
ax[0].set_xticks([0, 1], ["normal", "ulcer"]); ax[0].set_yticks([0, 1], ["normal", "ulcer"])
ax[0].set_xlabel("predicted"); ax[0].set_ylabel("true"); ax[0].set_title("Confusion matrix (test fold)")
fpr, tpr, _ = roc_curve(y_test, p_test)
ax[1].plot(fpr, tpr, label=f"AUC = {metrics['auc']:.3f}"); ax[1].plot([0, 1], [0, 1], "k--", lw=0.8)
ax[1].set_xlabel("false positive rate"); ax[1].set_ylabel("true positive rate"); ax[1].set_title("ROC curve"); ax[1].legend()
plt.tight_layout(); plt.show()

wrong = np.where(pred != y_test)[0][:8]
if len(wrong):
    plt.figure(figsize=(2 * len(wrong), 2.6))
    for k, i in enumerate(wrong):
        plt.subplot(1, len(wrong), k + 1); plt.imshow(X[idx["test"][i]]); plt.axis("off")
        plt.title(f"true {'ulcer' if y_test[i] else 'normal'}\np={p_test[i]:.2f}", fontsize=8)
    plt.suptitle("Misclassified test images"); plt.tight_layout(); plt.show()

## 6. Size and speed

In [ ]:
model.save("model.keras")
size_mb = model.count_params() * 4 / 1e6          # FP32 weights = 4 bytes per parameter (file sizes also include optimizer state)

def latency_ms(device, runs=100):
    # Single-image inference time with a compiled graph (excludes Python/eager overhead)
    with tf.device(device):
        infer = tf.function(lambda t: model(t, training=False))
        x = tf.constant(X[idx["test"][:1]].astype(np.float32))
        for _ in range(10): infer(x).numpy()                       # warm-up + graph compilation
        t = time.perf_counter()
        for _ in range(runs): infer(x).numpy()
        return (time.perf_counter() - t) / runs * 1000

efficiency = {
    "params": int(model.count_params()),
    "size_mb_fp32": round(size_mb, 2),
    "latency_cpu_ms": round(latency_ms("/CPU:0"), 2),
    "latency_gpu_ms": round(latency_ms("/GPU:0"), 2) if tf.config.list_physical_devices("GPU") else None,
    "train_time_min": round(train_time / 60, 2),
}
print(pd.Series(efficiency).to_string())

In [ ]:
result = {"model": MODEL_NAME, "test_fold": TEST_FOLD, "val_fold": VAL_FOLD, "threshold": round(THRESHOLD, 2), "chosen_weights": chosen,
          "n_train": int(len(idx["train"])), "n_val": int(len(idx["val"])), "n_test": int(len(idx["test"])),
          **{k: round(float(v), 4) for k, v in metrics.items()}, **efficiency,
          "settings": {"img": IMG_SIZE, "batch": BATCH, "epochs_head": EPOCHS_HEAD, "epochs_ft": EPOCHS_FT,
                       "lr_head": LR_HEAD, "lr_ft": LR_FT, "patience": PATIENCE, "seed": SEED}}
json.dump(result, open("result.json", "w"), indent=2)
pd.DataFrame({"file": data.file[idx["test"]].values, "class": data["class"][idx["test"]].values,
              "label": y_test.astype(int), "prob_ulcer": p_test}).to_csv("test_predictions.csv", index=False)
print(json.dumps(result, indent=2))

## What to check
- **Training curves:** validation loss should go down and then flatten. A validation loss rising while training loss falls would mean overfitting.
- **Test metrics:** the first honest result on unseen, leakage-free data. Don't compare it with 99% papers, because their evaluation was different.
- **Training time:** multiply by 30 (6 models x 5 folds) to estimate the full benchmark time.

If everything runs cleanly, notebook 05 reuses this exact code for all 6 models and all 5 folds.